# VM-local MLP Classification and Unlearning Pipeline

Pipeline chạy trực tiếp bằng CPU/GPU và ổ đĩa local của VM. Raw PCAP chỉ đọc; feature cache, checkpoint, log và summary được ghi vào workspace local của VM.

```text
.pcap -> packet features [relative_time, direction, packet_size] -> SupCon encoder -> embedding 256 -> MLP known/unknown -> checkpoint -> optional unlearning
```


In [ ]:
# OVERVIEW: Chọn hai dataset trên ổ local VM và một workspace local cho cache/artifact.
import os
from pathlib import Path

# Mỗi VM/run phải dùng RUN_ID riêng để không cùng ghi cache hoặc checkpoint.
RUN_ID = 'aol_known_vs_273_unknown_seed42'

# TODO(user): thay đúng HAI đường dẫn này bằng đường dẫn dataset trên ổ local của VM.
AOL_DATASET_DIR = Path('/CHANGE_ME/AOL')
UNKNOWN_273_DATASET_DIR = Path('/CHANGE_ME/273')

# Mặc định lấy checkpoint trong repository trên VM; đổi nếu bạn đặt checkpoint ở nơi khác.
PRETRAIN_AOL_PATH = Path('MLP-Classfication/vm_code/weight_trained/pretrain_AOL.pth')
if not PRETRAIN_AOL_PATH.exists():
    PRETRAIN_AOL_PATH = Path('weight_trained/pretrain_AOL.pth')
# Artifact local: không mount Drive và không ghi PCAP gốc.
EXPERIMENT_ROOT = Path.cwd() / 'artifacts' / 'vm-training' / 'experiments'
VM_OUTPUT_DIR = EXPERIMENT_ROOT / RUN_ID

os.environ['OUTPUT_DIR'] = str(VM_OUTPUT_DIR)
os.environ['DEVICE_NAME'] = 'cuda:0'

print('RUN_ID:', RUN_ID)
print('AOL_DATASET_DIR:', AOL_DATASET_DIR)
print('UNKNOWN_273_DATASET_DIR:', UNKNOWN_273_DATASET_DIR)
print('PRETRAIN_AOL_PATH:', PRETRAIN_AOL_PATH)
print('OUTPUT_DIR:', VM_OUTPUT_DIR)


In [ ]:
# OVERVIEW: Chuẩn bị dependency tối thiểu để parse PCAP và train PyTorch pipeline.
import importlib.util
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is not None:
        return
    package_name = pip_name or import_name
    print(f'Cài package còn thiếu: {package_name}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])


ensure_package('scapy')
print('Dependency check done.')

In [ ]:
# OVERVIEW: Cấu hình dữ liệu local, cache/artifact local, encoder, MLP và training.
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
# OUTPUT_DIR được đặt tại cell đầu và mặc định nằm trong ./artifacts của VM.
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', PROJECT_ROOT / 'artifacts' / 'vm-training')).expanduser()
FEATURE_CACHE_DIR = OUTPUT_DIR / 'feature_cache'

AOL_DATASET_DIR = Path(AOL_DATASET_DIR).expanduser()
UNKNOWN_273_DATASET_DIR = Path(UNKNOWN_273_DATASET_DIR).expanduser()
PRETRAIN_AOL_PATH = Path(PRETRAIN_AOL_PATH).expanduser()

# Legacy SupCon checkpoint contract from supcon-model.ipynb.
# Do not change these values when loading pretrain_AOL.pth.
MAX_PACKETS = 10000
PACKET_FEATURES = 3
FEATURE_TRANSFORM = 'legacy_raw_v1'
FEATURE_CACHE_VERSION = f'legacy_pcap3_{FEATURE_TRANSFORM}_max{MAX_PACKETS}_v1'

# Label split.
SEED = 42
MIN_SAMPLES_PER_LABEL = 10
MAX_FILES_PER_LABEL = None
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SPLIT_MANIFEST_PATH = OUTPUT_DIR / 'split_manifest.json'
LABEL_INVENTORY_PATH = OUTPUT_DIR / 'label_inventory.json'

# Model and training.
EMBEDDING_DIM = 256
PROJECTION_DIM = 128
HIDDEN_DIMS = (512, 256, 128, 64, 32)
OUTPUT_DIM = 2  # binary known/unknown; đổi thành num_classes khi mở rộng multiclass.
DROPOUT = 0.20
BATCH_SIZE = 16
NUM_WORKERS = 0

FREEZE_ENCODER_FOR_BASE_TRAINING = True

EPOCHS = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
LOG_EVERY_N_BATCHES = 2
BINARY_CHECKPOINT_EVERY_N_BATCHES = 2
RESUME_BINARY_FROM_CHECKPOINT = False
USE_CLASS_WEIGHTS = True
CHECKPOINT_SCORE_METRIC = 'balanced_accuracy'
UNKNOWN_THRESHOLD = 0.50
THRESHOLD_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

DEVICE_NAME = os.environ.get('DEVICE_NAME', '').strip()  # '' = auto
REQUIRE_CUDA = True

BASE_MODEL_DIR = OUTPUT_DIR / 'base_model'
BASE_MODEL_README_PATH = BASE_MODEL_DIR / 'README.md'
BINARY_TRAINING_LATEST_PATH = BASE_MODEL_DIR / 'binary_training_latest.pt'
BINARY_TRAINING_BEST_PATH = BASE_MODEL_DIR / 'binary_training_best.pt'
BEST_MODEL_PATH = BASE_MODEL_DIR / 'best_model.pt'
RUN_SUMMARY_JSON_PATH = OUTPUT_DIR / 'run_summary.json'
RUN_SUMMARY_MD_PATH = OUTPUT_DIR / 'run_summary.md'

# Phase 3. Để False cho lần base training đầu tiên; bật True để chạy cả ba baseline.
RUN_UNLEARNING = False
FORGET_LABEL = None  # None = tự chọn deterministic một AOL label có train/val/test.
UNLEARNING_EPOCHS = 3
UNLEARNING_LR = 1e-4
RETAIN_LOSS_WEIGHT = 1.0
UNLEARNING_SCOPES = ('head_only', 'last_encoder_block', 'full_encoder_and_head')
UNLEARNING_ROOT = OUTPUT_DIR / 'unlearning'

In [ ]:
# OVERVIEW: Nạp thư viện, tạo thư mục local output/cache và chọn thiết bị train.
from __future__ import annotations

import copy
import hashlib
import itertools
import json
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scapy.all import IP, IPv6, PcapReader
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE_NAME:
    DEVICE = torch.device(DEVICE_NAME)
else:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if REQUIRE_CUDA and DEVICE.type != 'cuda':
    raise RuntimeError(
        'Cấu hình hiện tại yêu cầu CUDA. Hãy chọn GPU runtime hoặc đặt REQUIRE_CUDA=False để debug CPU.'
    )

print('AOL_DATASET_DIR:', AOL_DATASET_DIR)
print('UNKNOWN_273_DATASET_DIR:', UNKNOWN_273_DATASET_DIR)
print('PRETRAIN_AOL_PATH:', PRETRAIN_AOL_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('FEATURE_CACHE_DIR:', FEATURE_CACHE_DIR)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device:', DEVICE)
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))

In [ ]:
# OVERVIEW: Preflight hai dataset local AOL/273, checkpoint legacy, workspace local và CUDA.
for dataset_name, dataset_dir in {'AOL': AOL_DATASET_DIR, '273_unknown': UNKNOWN_273_DATASET_DIR}.items():
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy dataset {dataset_name}: {dataset_dir}. Hãy sửa cell mount/config.')
if not PRETRAIN_AOL_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy pretrain_AOL.pth: {PRETRAIN_AOL_PATH}')
if DEVICE.type != 'cuda' or not torch.cuda.is_available():
    raise RuntimeError('VM không có CUDA khả dụng; dừng trước khi tạo cache/train.')

preflight_dir = OUTPUT_DIR / 'preflight'
preflight_dir.mkdir(parents=True, exist_ok=True)
probe_path = preflight_dir / 'local_write_probe.txt'
probe_tmp = probe_path.with_suffix('.tmp')
probe_tmp.write_text('Local VM write check passed\n', encoding='utf-8')
probe_tmp.replace(probe_path)

gpu = torch.cuda.get_device_properties(DEVICE)
pcap_counts = {
    name: sum(1 for path in dataset_dir.rglob('*') if path.is_file() and path.suffix.lower() in {'.pcap', '.cap', '.pcapng'})
    for name, dataset_dir in {'AOL': AOL_DATASET_DIR, '273_unknown': UNKNOWN_273_DATASET_DIR}.items()
}
if not all(pcap_counts.values()):
    raise FileNotFoundError(f'Mỗi dataset phải có PCAP/CAP/PCAPNG. Counts={pcap_counts}')

print('Local write probe:', probe_path)
print('PCAP/CAP/PCAPNG:', pcap_counts)
print('GPU:', gpu.name)
print(f'GPU memory: {gpu.total_memory / 1024**3:.2f} GiB')
print('CUDA:', torch.version.cuda)
print('Preflight passed.')


In [ ]:
# OVERVIEW: Định nghĩa record dữ liệu, seed, JSON helper, scan PCAP và split label ở cấp class.
@dataclass(frozen=True)
class FlowRecord:
    path: Path
    original_label: str
    binary_label: int  # 0 = known, 1 = unknown
    source: str = 'unspecified'


CLASS_NAMES = {0: 'known', 1: 'unknown'}
PCAP_SUFFIXES = {'.pcap', '.cap', '.pcapng'}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))


def scan_pcap_files(data_dirs: Sequence[Path]) -> list[tuple[Path, str]]:
    files: list[tuple[Path, str]] = []
    for data_dir in data_dirs:
        if not data_dir.exists():
            print(f'Bỏ qua DATA_DIR không tồn tại: {data_dir}')
            continue
        for path in sorted(data_dir.rglob('*')):
            if path.is_file() and path.suffix.lower() in PCAP_SUFFIXES:
                files.append((path, path.parent.name))
    if not files:
        raise FileNotFoundError(f'Không tìm thấy PCAP trong DATA_DIRS={data_dirs}')
    return files


def build_or_load_label_split(label_counts: Counter[str]) -> dict[str, list[str]]:
    if AUTO_LABEL_SPLIT and LABEL_SPLIT_PATH.exists() and not REBUILD_LABEL_SPLIT:
        payload = read_json(LABEL_SPLIT_PATH)
        print('Đọc label split:', LABEL_SPLIT_PATH)
        return {
            'known_labels': list(payload['known_labels']),
            'unknown_labels': list(payload['unknown_labels']),
            'holdout_unknown_labels': list(payload.get('holdout_unknown_labels', [])),
        }

    eligible = sorted(label for label, count in label_counts.items() if count >= MIN_SAMPLES_PER_LABEL)
    if len(eligible) < KNOWN_LABEL_COUNT + HOLDOUT_UNKNOWN_LABEL_COUNT + 1:
        raise ValueError(
            'Không đủ label để split. Cần ít nhất KNOWN_LABEL_COUNT + HOLDOUT_UNKNOWN_LABEL_COUNT + 1 label đủ sample.'
        )

    rng = random.Random(SEED)
    rng.shuffle(eligible)
    known_labels = sorted(eligible[:KNOWN_LABEL_COUNT])
    holdout_start = KNOWN_LABEL_COUNT
    holdout_end = holdout_start + HOLDOUT_UNKNOWN_LABEL_COUNT
    holdout_unknown_labels = sorted(eligible[holdout_start:holdout_end])
    unknown_labels = sorted(eligible[holdout_end:])

    payload = {
        'seed': SEED,
        'min_samples_per_label': MIN_SAMPLES_PER_LABEL,
        'known_label_count': KNOWN_LABEL_COUNT,
        'holdout_unknown_label_count': HOLDOUT_UNKNOWN_LABEL_COUNT,
        'known_labels': known_labels,
        'unknown_labels': unknown_labels,
        'holdout_unknown_labels': holdout_unknown_labels,
        'label_counts': {label: int(label_counts[label]) for label in sorted(eligible)},
    }
    write_json(LABEL_SPLIT_PATH, payload)
    print('Tạo label split:', LABEL_SPLIT_PATH)
    return payload


def make_records(
    files: Sequence[tuple[Path, str]],
    known_labels: set[str],
    unknown_labels: set[str],
    holdout_unknown_labels: set[str],
) -> list[FlowRecord]:
    selected = known_labels | unknown_labels | holdout_unknown_labels
    grouped: dict[str, list[Path]] = defaultdict(list)
    for path, label in files:
        if label in selected:
            grouped[label].append(path)

    rng = random.Random(SEED)
    records: list[FlowRecord] = []
    for label, paths in sorted(grouped.items()):
        paths = sorted(paths)
        rng.shuffle(paths)
        if MAX_FILES_PER_LABEL is not None:
            paths = paths[:MAX_FILES_PER_LABEL]
        binary_label = 0 if label in known_labels else 1
        records.extend(FlowRecord(path=path, original_label=label, binary_label=binary_label) for path in paths)
    return records


def split_records_by_label(
    records: Sequence[FlowRecord],
    holdout_unknown_labels: set[str],
    val_ratio: float,
    test_ratio: float,
) -> tuple[list[FlowRecord], list[FlowRecord], list[FlowRecord]]:
    rng = random.Random(SEED)
    by_label: dict[str, list[FlowRecord]] = defaultdict(list)
    for record in records:
        by_label[record.original_label].append(record)

    train: list[FlowRecord] = []
    val: list[FlowRecord] = []
    test: list[FlowRecord] = []

    for label, items in sorted(by_label.items()):
        items = list(items)
        rng.shuffle(items)
        n = len(items)
        n_test = max(1, int(round(n * test_ratio))) if n >= 3 else 0
        n_val = max(1, int(round(n * val_ratio))) if n - n_test >= 3 else 0

        if label in holdout_unknown_labels:
            val.extend(items[:n_val])
            test.extend(items[n_val:])
            continue

        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])

    return train, val, test


def count_by_binary(records: Sequence[FlowRecord]) -> dict[str, int]:
    counts = Counter(record.binary_label for record in records)
    return {'known': int(counts.get(0, 0)), 'unknown': int(counts.get(1, 0))}


def count_by_label(records: Sequence[FlowRecord]) -> dict[str, int]:
    return dict(sorted(Counter(record.original_label for record in records).items()))

In [ ]:
# OVERVIEW: Parse PCAP thành tensor [MAX_PACKETS, 3] và cache feature theo file/version.
def packet_rows(path: Path) -> list[tuple[float, str, str, int]]:
    rows: list[tuple[float, str, str, int]] = []
    try:
        with PcapReader(str(path)) as reader:
            for packet in reader:
                ip_layer = None
                if IP in packet:
                    ip_layer = packet[IP]
                elif IPv6 in packet:
                    ip_layer = packet[IPv6]
                if ip_layer is None:
                    continue
                rows.append((float(packet.time), str(ip_layer.src), str(ip_layer.dst), int(len(packet))))
    except Exception as error:
        print(f'Lỗi parse {path}: {type(error).__name__}: {error}')
    return rows


def infer_local_ip(rows: Sequence[tuple[float, str, str, int]]) -> str | None:
    # Khớp get_local_ip() của supcon-model.ipynb: đếm cả src và dst.
    addresses = [address for _time, src, dst, _size in rows for address in (src, dst)]
    return Counter(addresses).most_common(1)[0][0] if addresses else None


def transform_packet_feature(relative_time: float, direction: float, packet_size: float) -> tuple[float, float, float]:
    if FEATURE_TRANSFORM == 'legacy_raw_v1':
        return float(relative_time), float(direction), float(packet_size)
    raise ValueError(f'FEATURE_TRANSFORM không hỗ trợ: {FEATURE_TRANSFORM}')


def pcap_to_features(path: Path, max_packets: int) -> tuple[Tensor, Tensor]:
    rows = packet_rows(path)
    features = torch.zeros((max_packets, PACKET_FEATURES), dtype=torch.float32)
    mask = torch.zeros((max_packets,), dtype=torch.bool)
    if not rows:
        return features, mask

    local_ip = infer_local_ip(rows)
    start_time = rows[0][0]
    for index, (timestamp, src, _dst, packet_size) in enumerate(rows[:max_packets]):
        relative_time = max(0.0, timestamp - start_time)
        direction = 0.0 if local_ip is not None and src == local_ip else 1.0
        features[index] = torch.tensor(
            transform_packet_feature(relative_time, direction, float(packet_size)),
            dtype=torch.float32,
        )
        mask[index] = True
    return features, mask


def feature_cache_path(path: Path) -> Path:
    stat = path.stat()
    cache_key = hashlib.sha1(
        f'{path.resolve()}::{stat.st_size}::{int(stat.st_mtime)}::{FEATURE_CACHE_VERSION}'.encode('utf-8')
    ).hexdigest()
    return FEATURE_CACHE_DIR / f'{cache_key}.pt'


def load_or_parse_features(path: Path) -> tuple[Tensor, Tensor]:
    cache_path = feature_cache_path(path)
    if cache_path.exists():
        payload = torch.load(cache_path, map_location='cpu', weights_only=False)
        return payload['features'], payload['mask']
    features, mask = pcap_to_features(path, MAX_PACKETS)
    torch.save({'features': features, 'mask': mask}, cache_path)
    return features, mask

In [ ]:
# OVERVIEW: Định nghĩa Dataset/DataLoader cho binary classifier và SupCon original-label training.
class FlowDataset(Dataset):
    def __init__(self, records: Sequence[FlowRecord]):
        self.records = list(records)
        self.labels = torch.tensor([record.binary_label for record in self.records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str, str]:
        record = self.records[index]
        features, mask = load_or_parse_features(record.path)
        return (
            features,
            mask,
            torch.tensor(record.binary_label, dtype=torch.long),
            record.original_label,
            str(record.path),
        )


class OriginalLabelFlowDataset(Dataset):
    def __init__(self, records: Sequence[FlowRecord], label_to_id: dict[str, int]):
        self.records = list(records)
        self.label_to_id = dict(label_to_id)
        self.labels = torch.tensor([self.label_to_id[record.original_label] for record in self.records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str, str]:
        record = self.records[index]
        features, mask = load_or_parse_features(record.path)
        label = self.label_to_id[record.original_label]
        return features, mask, torch.tensor(label, dtype=torch.long), record.original_label, str(record.path)


def make_loader(records: Sequence[FlowRecord], shuffle: bool) -> DataLoader:
    return DataLoader(
        FlowDataset(records),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda',
    )


def make_original_label_loader(records: Sequence[FlowRecord], label_to_id: dict[str, int], shuffle: bool) -> DataLoader:
    return DataLoader(
        OriginalLabelFlowDataset(records, label_to_id),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda',
    )


def move_batch(batch: Sequence[Any], device: torch.device) -> tuple[Tensor, Tensor, Tensor]:
    features, mask, labels = batch[:3]
    return features.to(device), mask.to(device), labels.to(device)

In [ ]:
# OVERVIEW: Định nghĩa DF-style encoder, MLP classifier và FlowModel end-to-end.
class FlowEncoder(nn.Module):
    def __init__(self, max_packets: int, embedding_dim: int):
        super().__init__()
        self.max_packets = max_packets
        kernel_size = 8
        pool_size = 8
        pool_stride = 4

        self.conv1 = nn.Conv1d(PACKET_FEATURES, 32, kernel_size, stride=1)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride=1)
        self.batch_norm1 = nn.BatchNorm1d(32)
        self.max_pool_1 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.dropout1 = nn.Dropout(0.10)

        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride=1)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride=1)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.max_pool_2 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.dropout2 = nn.Dropout(0.10)

        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride=1)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=1)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.max_pool_3 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.dropout3 = nn.Dropout(0.10)

        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride=1)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=1)
        self.batch_norm4 = nn.BatchNorm1d(256)
        self.max_pool_4 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.dropout4 = nn.Dropout(0.10)

        with torch.no_grad():
            dummy = torch.zeros(1, PACKET_FEATURES, max_packets)
            flat_dim = self._forward_convs(dummy).reshape(1, -1).shape[1]
        self.fc = nn.Linear(flat_dim, embedding_dim)
        self._init_weights()
        print(f'FlowEncoder flat_dim={flat_dim}, embedding_dim={embedding_dim}')

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _block(self, x: Tensor, conv_a: nn.Conv1d, conv_b: nn.Conv1d, bn: nn.BatchNorm1d, pool: nn.MaxPool1d, drop: nn.Dropout, first: bool = False) -> Tensor:
        x = F.pad(x, (3, 4))
        x = F.elu(conv_a(x)) if first else F.relu(conv_a(x))
        x = F.pad(x, (3, 4))
        x = F.elu(bn(conv_b(x))) if first else F.relu(bn(conv_b(x)))
        x = F.pad(x, (3, 4))
        x = pool(x)
        return drop(x)

    def _forward_convs(self, x: Tensor) -> Tensor:
        x = self._block(x, self.conv1, self.conv1_1, self.batch_norm1, self.max_pool_1, self.dropout1, first=True)
        x = self._block(x, self.conv2, self.conv2_2, self.batch_norm2, self.max_pool_2, self.dropout2)
        x = self._block(x, self.conv3, self.conv3_3, self.batch_norm3, self.max_pool_3, self.dropout3)
        x = self._block(x, self.conv4, self.conv4_4, self.batch_norm4, self.max_pool_4, self.dropout4)
        return x

    def forward(self, features: Tensor, mask: Tensor | None = None) -> Tensor:
        if mask is not None:
            features = features * mask.unsqueeze(-1).float()
        x = features.transpose(1, 2)
        x = self._forward_convs(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], output_dim: int, dropout: float):
        super().__init__()
        layers: list[nn.Module] = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, embedding: Tensor) -> Tensor:
        return self.net(embedding)


class FlowModel(nn.Module):
    def __init__(self, output_dim: int = OUTPUT_DIM):
        super().__init__()
        self.encoder = FlowEncoder(MAX_PACKETS, EMBEDDING_DIM)
        self.classifier = MLPClassifier(EMBEDDING_DIM, HIDDEN_DIMS, output_dim, DROPOUT)
        self.encoder_frozen = False

    def forward(self, features: Tensor, mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        logits = self.classifier(embedding)
        return logits, embedding


def load_pretrained_aol_encoder(model: FlowModel, checkpoint_path: Path) -> dict[str, Any]:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    config = dict(checkpoint.get('model_config', {}))
    expected = {'max_packets': MAX_PACKETS, 'hidden_size': EMBEDDING_DIM, 'embedding_size': PROJECTION_DIM}
    if any(config.get(key) != value for key, value in expected.items()):
        raise ValueError(f'Checkpoint config không tương thích. expected={expected}, found={config}')
    encoder_state = {key.removeprefix('encoder.'): value for key, value in checkpoint['model_state_dict'].items() if key.startswith('encoder.')}
    incompatible = model.encoder.load_state_dict(encoder_state, strict=True)
    assert not incompatible.missing_keys and not incompatible.unexpected_keys
    return {'path': str(checkpoint_path), 'epoch': int(checkpoint.get('epoch', -1)), 'model_config': config}


def freeze_encoder(model: FlowModel) -> None:
    for parameter in model.encoder.parameters():
        parameter.requires_grad = False
    model.encoder.eval()
    model.encoder_frozen = True
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    frozen = sum(parameter.numel() for parameter in model.encoder.parameters())
    print(f'Freeze encoder: frozen={frozen:,}, trainable={trainable:,}')

In [ ]:
# OVERVIEW: Định nghĩa metric, evaluate, class weights và checkpoint helper dùng chung.
def evaluate(model: FlowModel, loader: DataLoader, unknown_threshold: float = UNKNOWN_THRESHOLD) -> dict[str, Any]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total = 0
    correct = 0
    known_total = 0
    known_correct = 0
    unknown_total = 0
    unknown_correct = 0
    predicted_unknown_total = 0
    confusion = torch.zeros(OUTPUT_DIM, OUTPUT_DIM, dtype=torch.long)

    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, DEVICE)
            logits, _ = model(features, mask)
            loss = criterion(logits, labels)
            probabilities = torch.softmax(logits, dim=1)
            if OUTPUT_DIM == 2:
                predictions = (probabilities[:, 1] >= unknown_threshold).long()
            else:
                predictions = probabilities.argmax(dim=1)

            total_loss += loss.item() * labels.numel()
            total += labels.numel()
            correct += (predictions == labels).sum().item()

            known = labels == 0
            unknown = labels == 1
            predicted_unknown = predictions == 1
            known_total += known.sum().item()
            known_correct += ((predictions == labels) & known).sum().item()
            unknown_total += unknown.sum().item()
            unknown_correct += ((predictions == labels) & unknown).sum().item()
            predicted_unknown_total += predicted_unknown.sum().item()

            for true_label, predicted_label in zip(labels.detach().cpu(), predictions.detach().cpu()):
                if int(true_label) < OUTPUT_DIM and int(predicted_label) < OUTPUT_DIM:
                    confusion[int(true_label), int(predicted_label)] += 1

    accuracy = correct / max(total, 1)
    known_recall = known_correct / max(known_total, 1)
    unknown_recall = unknown_correct / max(unknown_total, 1)
    unknown_precision = unknown_correct / max(predicted_unknown_total, 1)
    balanced_accuracy = 0.5 * (known_recall + unknown_recall)

    return {
        'loss': total_loss / max(total, 1),
        'accuracy': accuracy,
        'known_recall': known_recall,
        'unknown_recall': unknown_recall,
        'unknown_precision': unknown_precision,
        'balanced_accuracy': balanced_accuracy,
        'unknown_threshold': unknown_threshold,
        'known_total': known_total,
        'unknown_total': unknown_total,
        'predicted_unknown_total': predicted_unknown_total,
        'confusion_matrix': confusion.tolist(),
    }


def metric_value(metrics: dict[str, Any], name: str) -> float:
    value = metrics.get(name, float('nan'))
    return float(value) if isinstance(value, (int, float)) else float('nan')


def build_class_weights(loader: DataLoader) -> Tensor | None:
    if not USE_CLASS_WEIGHTS or not hasattr(loader.dataset, 'labels'):
        return None
    labels = loader.dataset.labels.detach().cpu()
    counts = torch.bincount(labels, minlength=OUTPUT_DIM).float()
    if (counts == 0).any():
        print('Không dùng class weights vì thiếu class:', counts.tolist())
        return None
    weights = counts.sum() / (OUTPUT_DIM * counts)
    weights = weights / weights.mean()
    print('class_counts=', counts.int().tolist(), 'class_weights=', weights.tolist())
    return weights.to(DEVICE)


def to_cpu_object(value: Any) -> Any:
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu_object(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu_object(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu_object(item) for item in value)
    return value


def save_atomic_torch(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(to_cpu_object(payload), tmp_path)
    tmp_path.replace(path)


def model_config_payload() -> dict[str, Any]:
    return {
        'max_packets': MAX_PACKETS,
        'packet_features': PACKET_FEATURES,
        'feature_transform': FEATURE_TRANSFORM,
        'embedding_dim': EMBEDDING_DIM,
        'projection_dim': PROJECTION_DIM,
        'hidden_dims': list(HIDDEN_DIMS),
        'output_dim': OUTPUT_DIM,
        'dropout': DROPOUT,
        'encoder_architecture': 'RawPacketEncoder-compatible',
    }


def label_payload() -> dict[str, Any]:
    return {
        'known_labels': sorted(KNOWN_LABELS),
        'unknown_labels': sorted(UNKNOWN_LABELS),
        'known_source': 'AOL',
        'unknown_source': '273',
        'split_manifest_path': str(SPLIT_MANIFEST_PATH),
    }

In [ ]:
# OVERVIEW: Train MLP binary classifier, lưu latest/best checkpoint và hỗ trợ resume/warm-start.
def save_binary_checkpoint(
    path: Path,
    model: FlowModel,
    optimizer: torch.optim.Optimizer,
    history: list[dict[str, float]],
    best_state: dict[str, Tensor],
    best_score: float,
    epoch: int,
    batch_index: int,
    status: str,
    total_seconds: float,
) -> None:
    payload = {
        'checkpoint_type': 'binary_training',
        'status': status,
        'model_state_dict': model.state_dict(),
        'best_model_state_dict': best_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_score': float(best_score),
        'checkpoint_score_metric': CHECKPOINT_SCORE_METRIC,
        'unknown_threshold': UNKNOWN_THRESHOLD,
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'progress': {
            'epoch': int(epoch),
            'batch_index': int(batch_index),
            'total_seconds': float(total_seconds),
        },
    }
    save_atomic_torch(payload, path)
    print(f'Lưu binary checkpoint ({status}): {path}')


def maybe_load_binary_checkpoint(
    path: Path,
    model: FlowModel,
    optimizer: torch.optim.Optimizer,
) -> tuple[list[dict[str, float]], dict[str, Tensor], float, int]:
    best_state = copy.deepcopy(model.state_dict())
    if not RESUME_BINARY_FROM_CHECKPOINT or not path.exists():
        return [], best_state, -float('inf'), 1
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    history = list(checkpoint.get('history', []))
    best_state = checkpoint.get('best_model_state_dict') or copy.deepcopy(model.state_dict())
    best_score = float(checkpoint.get('best_score', -float('inf')))
    progress = checkpoint.get('progress', {})
    status = checkpoint.get('status', 'unknown')
    last_epoch = int(progress.get('epoch', 0))
    start_epoch = last_epoch + 1 if status in {'epoch_done', 'best', 'finished_best_loaded'} else max(last_epoch, 1)
    print(f'Resume binary training từ {path}, status={status}, start_epoch={start_epoch}')
    return history, best_state, best_score, start_epoch


def train_binary_classifier(model: FlowModel, train_loader: DataLoader, val_loader: DataLoader) -> dict[str, Any]:
    model.to(DEVICE)
    class_weights = build_class_weights(train_loader)
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable_parameters:
        raise ValueError('Không có tham số trainable. Kiểm tra freeze encoder/classifier.')
    optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    history, best_state, best_score, start_epoch = maybe_load_binary_checkpoint(BINARY_TRAINING_LATEST_PATH, model, optimizer)
    run_start = time.perf_counter()

    print(
        f'Binary train: epochs={EPOCHS}, batch_size={BATCH_SIZE}, train_batches={len(train_loader)}, '
        f'val_batches={len(val_loader)}, metric={CHECKPOINT_SCORE_METRIC}'
    )

    try:
        for epoch in range(start_epoch, EPOCHS + 1):
            model.train()
            if getattr(model, 'encoder_frozen', False):
                model.encoder.eval()
            epoch_start = time.perf_counter()
            total_loss = 0.0
            total = 0
            batches_seen = 0
            for batch_index, batch in enumerate(train_loader, start=1):
                features, mask, labels = move_batch(batch, DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits, _embedding = model(features, mask)
                loss = criterion(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(trainable_parameters, max_norm=5.0)
                optimizer.step()

                total_loss += loss.item() * labels.numel()
                total += labels.numel()
                batches_seen += 1
                if batch_index == 1 or batch_index % max(LOG_EVERY_N_BATCHES, 1) == 0:
                    print(f'Epoch {epoch:03d}/{EPOCHS} batch {batch_index:04d}/{len(train_loader)} loss={loss.item():.4f}')
                if batch_index == 1 or batch_index % max(BINARY_CHECKPOINT_EVERY_N_BATCHES, 1) == 0:
                    save_binary_checkpoint(
                        BINARY_TRAINING_LATEST_PATH,
                        model,
                        optimizer,
                        history,
                        best_state,
                        best_score,
                        epoch,
                        batch_index,
                        'running',
                        time.perf_counter() - run_start,
                    )

            val_metrics = evaluate(model, val_loader, UNKNOWN_THRESHOLD)
            score = metric_value(val_metrics, CHECKPOINT_SCORE_METRIC)
            if math.isnan(score):
                score = -float('inf')
            is_best = score > best_score
            if is_best:
                best_score = score
                best_state = copy.deepcopy(model.state_dict())

            row = {
                'epoch': float(epoch),
                'train_loss': total_loss / max(total, 1),
                'train_batches': float(batches_seen),
                'val_loss': float(val_metrics['loss']),
                'val_accuracy': float(val_metrics['accuracy']),
                'val_known_recall': float(val_metrics['known_recall']),
                'val_unknown_recall': float(val_metrics['unknown_recall']),
                'val_balanced_accuracy': float(val_metrics['balanced_accuracy']),
                'checkpoint_score': float(score),
                'epoch_seconds': time.perf_counter() - epoch_start,
            }
            history.append(row)
            print(
                f"Epoch {epoch:03d} done train_loss={row['train_loss']:.4f} "
                f"val_balanced_acc={row['val_balanced_accuracy']:.4f} confusion={val_metrics['confusion_matrix']}"
            )
            save_binary_checkpoint(
                BINARY_TRAINING_LATEST_PATH,
                model,
                optimizer,
                history,
                best_state,
                best_score,
                epoch,
                batches_seen,
                'epoch_done',
                time.perf_counter() - run_start,
            )
            if is_best:
                save_binary_checkpoint(
                    BINARY_TRAINING_BEST_PATH,
                    model,
                    optimizer,
                    history,
                    best_state,
                    best_score,
                    epoch,
                    batches_seen,
                    'best',
                    time.perf_counter() - run_start,
                )
    except KeyboardInterrupt:
        save_binary_checkpoint(
            BINARY_TRAINING_LATEST_PATH,
            model,
            optimizer,
            history,
            best_state,
            best_score,
            epoch if 'epoch' in locals() else 0,
            batch_index if 'batch_index' in locals() else 0,
            'interrupted',
            time.perf_counter() - run_start,
        )
        raise

    model.load_state_dict(best_state)
    total_seconds = time.perf_counter() - run_start
    save_binary_checkpoint(
        BINARY_TRAINING_LATEST_PATH,
        model,
        optimizer,
        history,
        best_state,
        best_score,
        EPOCHS,
        len(train_loader),
        'finished_best_loaded',
        total_seconds,
    )
    return {
        'history': history,
        'best_score': best_score,
        'total_seconds': total_seconds,
        'latest_checkpoint': str(BINARY_TRAINING_LATEST_PATH),
        'best_checkpoint': str(BINARY_TRAINING_BEST_PATH),
    }

In [ ]:
# OVERVIEW: Tạo binary dataset cố định: AOL=known, 273=unknown; split độc lập theo source+parent label.
seed_everything(SEED)
aol_files = scan_pcap_files([AOL_DATASET_DIR])
unknown_273_files = scan_pcap_files([UNKNOWN_273_DATASET_DIR])

def build_source_records(files: Sequence[tuple[Path, str]], source: str, binary_label: int) -> list[FlowRecord]:
    grouped: dict[str, list[Path]] = defaultdict(list)
    for path, label in files:
        grouped[label].append(path)
    records: list[FlowRecord] = []
    rng = random.Random(SEED)
    for label, paths in sorted(grouped.items()):
        if len(paths) < MIN_SAMPLES_PER_LABEL:
            continue
        paths = sorted(paths)
        rng.shuffle(paths)
        if MAX_FILES_PER_LABEL is not None:
            paths = paths[:MAX_FILES_PER_LABEL]
        records.extend(FlowRecord(path=path, original_label=label, binary_label=binary_label, source=source) for path in paths)
    return records

def split_by_source_and_label(records: Sequence[FlowRecord]) -> tuple[list[FlowRecord], list[FlowRecord], list[FlowRecord]]:
    grouped: dict[tuple[str, str], list[FlowRecord]] = defaultdict(list)
    for record in records:
        grouped[(record.source, record.original_label)].append(record)
    rng = random.Random(SEED)
    train: list[FlowRecord] = []; val: list[FlowRecord] = []; test: list[FlowRecord] = []
    for _key, items in sorted(grouped.items()):
        items = list(items); rng.shuffle(items); n = len(items)
        n_test = max(1, int(round(n * TEST_RATIO))) if n >= 3 else 0
        n_val = max(1, int(round(n * VAL_RATIO))) if n - n_test >= 3 else 0
        test.extend(items[:n_test]); val.extend(items[n_test:n_test + n_val]); train.extend(items[n_test + n_val:])
    return train, val, test

records = build_source_records(aol_files, 'AOL', 0) + build_source_records(unknown_273_files, '273', 1)
if not records or not any(record.binary_label == 0 for record in records) or not any(record.binary_label == 1 for record in records):
    raise ValueError('Cần có đủ AOL known và 273 unknown sau khi áp dụng MIN_SAMPLES_PER_LABEL.')
train_records, val_records, test_records = split_by_source_and_label(records)
KNOWN_LABELS = {record.original_label for record in records if record.source == 'AOL'}
UNKNOWN_LABELS = {record.original_label for record in records if record.source == '273'}
HOLDOUT_UNKNOWN_LABELS: set[str] = set()

inventory = {
    'aol_dataset_dir': str(AOL_DATASET_DIR), 'unknown_273_dataset_dir': str(UNKNOWN_273_DATASET_DIR),
    'binary_definition': {'AOL': 0, '273': 1},
    'counts_by_source': {'AOL': len(aol_files), '273': len(unknown_273_files)},
    'selected_counts_by_binary': count_by_binary(records),
    'aol_labels': sorted(KNOWN_LABELS), 'unknown_273_labels': sorted(UNKNOWN_LABELS),
}
write_json(LABEL_INVENTORY_PATH, inventory)
write_json(SPLIT_MANIFEST_PATH, {
    'seed': SEED, 'val_ratio': VAL_RATIO, 'test_ratio': TEST_RATIO,
    'records': [{'path': str(record.path), 'source': record.source, 'original_label': record.original_label, 'binary_label': record.binary_label,
                 'split': split} for split, split_records in [('train', train_records), ('val', val_records), ('test', test_records)] for record in split_records],
})
print('Binary definition: AOL=known (0), 273=unknown (1)')
print('Record counts:', {'total': count_by_binary(records), 'train': count_by_binary(train_records), 'val': count_by_binary(val_records), 'test': count_by_binary(test_records)})

train_loader = make_loader(train_records, shuffle=True)
val_loader = make_loader(val_records, shuffle=False)
test_loader = make_loader(test_records, shuffle=False)

model = FlowModel(output_dim=OUTPUT_DIM).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Model parameters: {parameter_count:,}')

In [ ]:
# OVERVIEW: Nạp encoder AOL đã pretrain, freeze encoder và train MLP binary bằng AOL known + 273 unknown.
pretrain_result = load_pretrained_aol_encoder(model, PRETRAIN_AOL_PATH)
print('Đã nạp encoder checkpoint:', pretrain_result)
if FREEZE_ENCODER_FOR_BASE_TRAINING:
    freeze_encoder(model)
else:
    print('Base training sẽ fine-tune cả encoder và MLPClassifier.')

training_result = train_binary_classifier(model, train_loader, val_loader)
print('Best validation score:', training_result['best_score'])

In [ ]:
# OVERVIEW: Sweep threshold trên validation, evaluate test, lưu best_model và run_summary.
def sweep_unknown_threshold(model: FlowModel, loader: DataLoader, grid: Sequence[float]) -> dict[str, Any]:
    rows = []
    best_row = None
    for threshold in grid:
        metrics = evaluate(model, loader, unknown_threshold=float(threshold))
        row = {'threshold': float(threshold), **metrics}
        rows.append(row)
        if best_row is None or row['balanced_accuracy'] > best_row['balanced_accuracy']:
            best_row = row
    assert best_row is not None
    return {'best_threshold': best_row['threshold'], 'best_metrics': best_row, 'rows': rows}


def save_final_model(path: Path, model: FlowModel, best_threshold: float, test_metrics: dict[str, Any]) -> None:
    payload = {
        'checkpoint_type': 'final_model',
        'model_state_dict': model.state_dict(),
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'best_unknown_threshold': float(best_threshold),
        'test_metrics': test_metrics,
    }
    save_atomic_torch(payload, path)
    print('Lưu final model:', path)


threshold_result = sweep_unknown_threshold(model, val_loader, THRESHOLD_GRID)
BEST_UNKNOWN_THRESHOLD = float(threshold_result['best_threshold'])
test_metrics = evaluate(model, test_loader, unknown_threshold=BEST_UNKNOWN_THRESHOLD)
save_final_model(BEST_MODEL_PATH, model, BEST_UNKNOWN_THRESHOLD, test_metrics)

summary = {
    'runtime': {
        'device': str(DEVICE),
        'torch_version': torch.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    'config': model_config_payload(),
    'paths': {
        'output_dir': str(OUTPUT_DIR),
        'feature_cache_dir': str(FEATURE_CACHE_DIR),
        'pretrain_aol': str(PRETRAIN_AOL_PATH),
        'binary_training_latest': str(BINARY_TRAINING_LATEST_PATH),
        'binary_training_best': str(BINARY_TRAINING_BEST_PATH),
        'best_model': str(BEST_MODEL_PATH),
    },
    'labels': label_payload(),
    'split': {
        'total_records': len(records),
        'train_records': len(train_records),
        'val_records': len(val_records),
        'test_records': len(test_records),
        'train_binary_counts': count_by_binary(train_records),
        'val_binary_counts': count_by_binary(val_records),
        'test_binary_counts': count_by_binary(test_records),
        'selected_label_counts': count_by_label(records),
    },
    'pretrain': pretrain_result,
    'training': training_result,
    'threshold_sweep': threshold_result,
    'test_metrics': test_metrics,
}
write_json(RUN_SUMMARY_JSON_PATH, summary)
write_json(BASE_MODEL_DIR / 'label_map.json', {'0': 'known_AOL', '1': 'unknown_273', 'aol_labels': sorted(KNOWN_LABELS), 'unknown_273_labels': sorted(UNKNOWN_LABELS)})
BASE_MODEL_README_PATH.write_text('\n'.join([
    '# Base binary model', '',
    '- Task: AOL = known (0); 273 = unknown (1).',
    '- Encoder: RawPacketEncoder loaded from pretrain_AOL.pth.',
    '- Input contract: [10000, 3] = [relative_time_seconds, direction, raw_packet_size].',
    '- Base training: encoder frozen=' + str(FREEZE_ENCODER_FOR_BASE_TRAINING) + '; MLPClassifier trained on both sources.',
    '- Best checkpoint: `best_model.pt`.',
    '- Split manifest: `../split_manifest.json`.',
    '- Do not treat this checkpoint as an unlearning result.',
]) + '\n', encoding='utf-8')

md_lines = [
    '# Run Summary',
    '',
    f'- device: `{summary["runtime"]["device"]}`',
    f'- cuda_available: `{summary["runtime"]["cuda_available"]}`',
    f'- total_records: `{len(records)}`',
    f'- train: `{len(train_records)}` {count_by_binary(train_records)}',
    f'- validation: `{len(val_records)}` {count_by_binary(val_records)}',
    f'- test: `{len(test_records)}` {count_by_binary(test_records)}',
    f'- best_unknown_threshold: `{BEST_UNKNOWN_THRESHOLD}`',
    f'- test_balanced_accuracy: `{test_metrics["balanced_accuracy"]}`',
    f'- test_confusion_matrix: `{test_metrics["confusion_matrix"]}`',
]
RUN_SUMMARY_MD_PATH.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
print('Lưu summary:', RUN_SUMMARY_JSON_PATH)
print('Lưu summary:', RUN_SUMMARY_MD_PATH)
print('Test metrics:', test_metrics)

In [ ]:
# OVERVIEW: Phase 3 class-level unlearning với ba baseline phạm vi cập nhật, mặc định không chạy.
# Df là toàn bộ AOL records thuộc một folder label; Dr là toàn bộ records còn lại (AOL khác + 273).
# Objective: CE(M(Df), unknown=1) + RETAIN_LOSS_WEIGHT * CE(M(Dr), label_gốc).

def choose_forget_label() -> str:
    candidates = []
    for label in sorted(KNOWN_LABELS):
        has_all_splits = all(any(record.source == 'AOL' and record.original_label == label for record in split) for split in (train_records, val_records, test_records))
        if has_all_splits:
            candidates.append(label)
    if not candidates:
        raise ValueError('Không có AOL label nào xuất hiện trong đủ train/validation/test để làm Df.')
    return random.Random(SEED).choice(candidates)


def split_forget_and_retain(records_pool: Sequence[FlowRecord], forget_label: str) -> tuple[list[FlowRecord], list[FlowRecord]]:
    forget = [record for record in records_pool if record.source == 'AOL' and record.original_label == forget_label]
    retain = [record for record in records_pool if record not in forget]
    return forget, retain


def set_unlearning_trainable_scope(model: FlowModel, scope: str) -> list[str]:
    for parameter in model.parameters():
        parameter.requires_grad = False
    if scope == 'head_only':
        modules = [model.classifier]
    elif scope == 'last_encoder_block':
        modules = [model.encoder.conv4, model.encoder.conv4_4, model.encoder.batch_norm4, model.encoder.fc, model.classifier]
    elif scope == 'full_encoder_and_head':
        modules = [model.encoder, model.classifier]
    else:
        raise ValueError(f'Unknown unlearning scope: {scope}')
    for module in modules:
        for parameter in module.parameters():
            parameter.requires_grad = True
    model.encoder_frozen = scope == 'head_only'
    return [name for name, parameter in model.named_parameters() if parameter.requires_grad]


def set_unlearning_train_mode(model: FlowModel, scope: str) -> None:
    model.train()
    if scope == 'head_only':
        model.encoder.eval()
    elif scope == 'last_encoder_block':
        # Giữ thống kê BatchNorm/Dropout của ba block đầu cố định; chỉ block 4 + FC được cập nhật.
        model.encoder.eval()
        for module in (model.encoder.conv4, model.encoder.conv4_4, model.encoder.batch_norm4, model.encoder.max_pool_4, model.encoder.dropout4, model.encoder.fc):
            module.train()
        model.classifier.train()


def forget_policy_metrics(model: FlowModel, forget_loader: DataLoader, unknown_threshold: float) -> dict[str, Any]:
    model.eval()
    total = 0
    predicted_unknown = 0
    probability_sum = 0.0
    with torch.no_grad():
        for batch in forget_loader:
            features, mask, _labels = move_batch(batch, DEVICE)
            logits, _ = model(features, mask)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            predicted_unknown += (probabilities >= unknown_threshold).sum().item()
            probability_sum += probabilities.sum().item()
            total += probabilities.numel()
    return {
        'total': total,
        'forget_as_unknown_rate': predicted_unknown / max(total, 1),
        'forget_mean_unknown_probability': probability_sum / max(total, 1),
        'unknown_threshold': float(unknown_threshold),
    }


def evaluate_unlearning_policy(model: FlowModel, forget_loader: DataLoader, retain_loader: DataLoader) -> dict[str, Any]:
    forget_metrics = forget_policy_metrics(model, forget_loader, BEST_UNKNOWN_THRESHOLD)
    retain_metrics = evaluate(model, retain_loader, BEST_UNKNOWN_THRESHOLD)
    policy_score = 0.5 * (forget_metrics['forget_as_unknown_rate'] + retain_metrics['balanced_accuracy'])
    return {'forget': forget_metrics, 'retain': retain_metrics, 'policy_score': policy_score}


def save_unlearning_artifacts(
    scope: str,
    forget_label: str,
    model: FlowModel,
    trainable_names: Sequence[str],
    validation: dict[str, Any],
    test: dict[str, Any],
    history: Sequence[dict[str, Any]],
) -> dict[str, str]:
    artifact_dir = UNLEARNING_ROOT / scope
    checkpoint_path = artifact_dir / 'unlearned_model.pt'
    result_path = artifact_dir / 'result.json'
    readme_path = artifact_dir / 'README.md'
    payload = {
        'checkpoint_type': 'class_level_unlearning',
        'baseline_scope': scope,
        'base_model_path': str(BEST_MODEL_PATH),
        'pretrain_encoder_path': str(PRETRAIN_AOL_PATH),
        'forget_source': 'AOL',
        'forget_label': forget_label,
        'forget_target_binary_label': 1,
        'loss': 'CE(Df, unknown=1) + lambda * CE(Dr, original_binary_label)',
        'retain_loss_weight': RETAIN_LOSS_WEIGHT,
        'model_state_dict': model.state_dict(),
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'trainable_parameter_names': list(trainable_names),
        'validation': validation,
        'test': test,
        'history': list(history),
    }
    save_atomic_torch(payload, checkpoint_path)
    write_json(result_path, {key: value for key, value in payload.items() if key != 'model_state_dict'})
    readme_path.parent.mkdir(parents=True, exist_ok=True)
    readme_path.write_text('\n'.join([
        f'# Unlearning baseline: {scope}', '',
        '- Base task: AOL = known (0), 273 = unknown (1).',
        '- Forget set Df: AOL folder label `' + forget_label + '`; its desired output after unlearning is `unknown` (1).',
        '- Retain set Dr: all remaining AOL and all 273 records, with original binary labels.',
        '- Loss: `CE(Df, unknown=1) + lambda * CE(Dr, original_binary_label)`; lambda=' + str(RETAIN_LOSS_WEIGHT) + '.',
        '- Trainable scope: `' + scope + '`.',
        '- Base model: `../../base_model/best_model.pt`.',
        '- This is a controlled relabel-to-unknown baseline, not a proof of certified unlearning.',
]) + '\n', encoding='utf-8')
    return {'artifact_dir': str(artifact_dir), 'checkpoint': str(checkpoint_path), 'result': str(result_path), 'readme': str(readme_path)}


def run_unlearning_baseline(
    scope: str,
    forget_label: str,
    forget_train_loader: DataLoader,
    retain_train_loader: DataLoader,
    forget_val_loader: DataLoader,
    retain_val_loader: DataLoader,
    forget_test_loader: DataLoader,
    retain_test_loader: DataLoader,
) -> dict[str, Any]:
    branch = copy.deepcopy(model).to(DEVICE)
    trainable_names = set_unlearning_trainable_scope(branch, scope)
    trainable_parameters = [parameter for parameter in branch.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable_parameters, lr=UNLEARNING_LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_state = copy.deepcopy(branch.state_dict())
    best_validation: dict[str, Any] | None = None
    best_score = -float('inf')
    history: list[dict[str, Any]] = []
    steps_per_epoch = max(len(forget_train_loader), len(retain_train_loader))

    for epoch in range(1, UNLEARNING_EPOCHS + 1):
        set_unlearning_train_mode(branch, scope)
        forget_iter = itertools.cycle(forget_train_loader)
        retain_iter = itertools.cycle(retain_train_loader)
        total_loss = total_forget_loss = total_retain_loss = 0.0
        for _step in range(steps_per_epoch):
            forget_features, forget_mask, _forget_labels = move_batch(next(forget_iter), DEVICE)
            retain_features, retain_mask, retain_labels = move_batch(next(retain_iter), DEVICE)
            optimizer.zero_grad(set_to_none=True)
            forget_logits, _ = branch(forget_features, forget_mask)
            retain_logits, _ = branch(retain_features, retain_mask)
            forget_targets = torch.ones(forget_logits.shape[0], dtype=torch.long, device=DEVICE)
            forget_loss = criterion(forget_logits, forget_targets)
            retain_loss = criterion(retain_logits, retain_labels)
            loss = forget_loss + RETAIN_LOSS_WEIGHT * retain_loss
            loss.backward()
            nn.utils.clip_grad_norm_(trainable_parameters, max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()
            total_forget_loss += forget_loss.item()
            total_retain_loss += retain_loss.item()

        validation = evaluate_unlearning_policy(branch, forget_val_loader, retain_val_loader)
        score = float(validation['policy_score'])
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(branch.state_dict())
            best_validation = validation
        row = {
            'epoch': epoch,
            'loss': total_loss / max(steps_per_epoch, 1),
            'forget_loss': total_forget_loss / max(steps_per_epoch, 1),
            'retain_loss': total_retain_loss / max(steps_per_epoch, 1),
            'validation_policy_score': score,
            'validation_forget_as_unknown_rate': validation['forget']['forget_as_unknown_rate'],
            'validation_retain_balanced_accuracy': validation['retain']['balanced_accuracy'],
        }
        history.append(row)
        print(f'Unlearning {scope} epoch {epoch}/{UNLEARNING_EPOCHS}: loss={row["loss"]:.4f}, forget->unknown={row["validation_forget_as_unknown_rate"]:.4f}, retain_bal_acc={row["validation_retain_balanced_accuracy"]:.4f}')

    branch.load_state_dict(best_state)
    assert best_validation is not None
    test = evaluate_unlearning_policy(branch, forget_test_loader, retain_test_loader)
    paths = save_unlearning_artifacts(scope, forget_label, branch, trainable_names, best_validation, test, history)
    return {'scope': scope, 'best_validation': best_validation, 'test': test, 'history': history, 'paths': paths}


if RUN_UNLEARNING:
    selected_forget_label = FORGET_LABEL or choose_forget_label()
    forget_train, retain_train = split_forget_and_retain(train_records, selected_forget_label)
    forget_val, retain_val = split_forget_and_retain(val_records, selected_forget_label)
    forget_test, retain_test = split_forget_and_retain(test_records, selected_forget_label)
    if not all((forget_train, retain_train, forget_val, retain_val, forget_test, retain_test)):
        raise ValueError(f'Df/Dr rỗng cho forget label={selected_forget_label}; hãy chọn AOL label khác.')
    print(f'Forget label: {selected_forget_label}; Df train/val/test={len(forget_train)}/{len(forget_val)}/{len(forget_test)}; Dr={len(retain_train)}/{len(retain_val)}/{len(retain_test)}')
    results = [
        run_unlearning_baseline(
            scope, selected_forget_label,
            make_loader(forget_train, True), make_loader(retain_train, True),
            make_loader(forget_val, False), make_loader(retain_val, False),
            make_loader(forget_test, False), make_loader(retain_test, False),
        )
        for scope in UNLEARNING_SCOPES
    ]
    write_json(UNLEARNING_ROOT / 'summary.json', {'forget_label': selected_forget_label, 'results': results})
    print('Lưu unlearning summary:', UNLEARNING_ROOT / 'summary.json')
else:
    print('RUN_UNLEARNING=False: đã train/lưu base model; đặt True để chạy ba baseline class-level unlearning.')